In [8]:
import os

import mlflow
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from setup.plotly_theme import register_theme

register_theme()

PDF_FOLDER = ".local/pdf"
os.makedirs(PDF_FOLDER, exist_ok=True)

In [9]:
# --- change these to look at another run / iteration / front ---
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "Rev-Benchmark_convergence_Opt-15-09-2026"
MLFLOW_RUN = "Run_at_2026-10-01_10-21-54"
ITERATION = 4
FRONT = "pareto_front"  # or "manufacturing_front"

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)


def get_run(experiment_name, run_name):
    experiment = client.get_experiment_by_name(experiment_name)
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName='{run_name}'",
    )
    if not runs:
        raise ValueError(f"Run {run_name} not found in experiment {experiment_name}")
    return runs[0]


def load_artifact(run, iteration, filename):
    path = client.download_artifacts(run.info.run_id, f"Iteration_{iteration:03d}/{filename}")
    return pl.read_csv(path)


def load_convergence(run, iteration, front="pareto_front"):
    """Load the NSGA2 convergence history with best_fN renamed to the objective labels.

    pymoo minimises, so maximised objectives are negated back to their real values.
    """
    convergence = load_artifact(run, iteration, f"{front}_convergence.csv")
    front_df = load_artifact(run, iteration, f"{front}_predictions.csv")
    objectives = [c for c in front_df.columns if c.startswith("obj:")]
    convergence = convergence.with_columns(
        [
            (-pl.col(f"best_f{i + 1}") if obj.startswith("obj:max") else pl.col(f"best_f{i + 1}")).alias(obj)
            for i, obj in enumerate(objectives)
        ]
    ).drop([f"best_f{i + 1}" for i in range(len(objectives))])
    return convergence, objectives


run = get_run(MLFLOW_EXPERIMENT, MLFLOW_RUN)
convergence, objectives = load_convergence(run, ITERATION, FRONT)
convergence

n_gen,n_evals,hypervolume,obj:max(μ(product_concentration)),obj:max(σ(product_concentration)),obj:max(R(product_concentration)),obj:max(P(purity))
i64,i64,f64,f64,f64,f64,f64
1,180,1.9693e9,2462.722302,448.434489,2459.203458,1.0
2,360,2.2872e9,2555.967879,448.434489,2551.479622,1.0
3,540,2.7279e9,2555.967879,459.266702,2551.479622,1.0
4,720,2.9047e9,2555.967879,459.266702,2551.479622,1.0
5,900,3.1031e9,2559.756933,494.346398,2560.283483,1.0
…,…,…,…,…,…,…
296,53280,3.9272e9,2622.292909,575.020279,2620.615458,1.0
297,53460,3.9043e9,2622.292909,575.020279,2620.615458,1.0
298,53640,3.8805e9,2622.292909,575.020279,2620.615458,1.0


In [10]:
def plot_convergence(convergence, objectives, title, filename=None):
    """Hypervolume plus best value of each objective per NSGA2 generation."""
    n_rows = 1 + len(objectives)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=["Hypervolume"] + [f"Best {obj.removeprefix('obj:')}" for obj in objectives],
    )
    for row, column in enumerate(["hypervolume"] + objectives, start=1):
        fig.add_trace(
            go.Scatter(
                x=convergence["n_gen"],
                y=convergence[column],
                mode="lines",
                line=dict(width=2, color="#0072B2"),
                showlegend=False,
            ),
            row=row,
            col=1,
        )
    fig.update_xaxes(title_text="Generation", row=n_rows, col=1)
    fig.update_layout(title=title, height=220 * n_rows, width=800)
    if filename:
        fig.write_image(f"{PDF_FOLDER}/{filename}.pdf")
    fig.show()
    return fig


fig = plot_convergence(
    convergence,
    objectives,
    title=f"NSGA2 Convergence ({FRONT.replace('_', ' ')}, iteration {ITERATION})",
)
fig.write_image(f"{PDF_FOLDER}/convergence_{FRONT}_iteration_{ITERATION}.pdf")